<a href="https://colab.research.google.com/github/nikkkk-kkkkhil/project1/blob/main/TinyGPT_English_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU name:", torch.cuda.get_device_name(0))
  device = "cuda"
else:
  device = "cpu"

print("Training device:", device)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4
Training device: cuda


In [6]:
text = """
Artificial intelligence is a field of computer science.
A language model learns patterns from the text.
A neural network contains learnable parameters.
The model predicts the next token in a sequence.
Training improves predictions by updating model weights.
Python is useful for machine learning.
Data quality matters when training a language model.
A transformer uses attention to process sequences.
The model learns from examples, not from human intution.
"""

import re

# Normalize whitespace
text = re.sub(r"\s+"," ", text).strip()

# Basic dataset checks
assert len(text)>0, "Dataset is empty!"
assert len(set(text))>1, "Dataset needs more variety!"

from pathlib import Path

data_dir = Path("/content/tinygpt_data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir/"english.txt"
dataset_path.write_text(text, encoding="utf-8")

print("Dataset saved to:", dataset_path)
print("File size:", dataset_path.stat().st_size, "bytes")


Dataset saved to: /content/tinygpt_data/english.txt
File size: 457 bytes


In [8]:
import torch
from pathlib import Path

dataset_path = Path("/content/tinygpt_data/english.txt")
loaded_text = dataset_path.read_text(encoding="utf-8")

print("PROJECT:", "TinyGPT-English")
print("PYTORCH:", torch.__version__)
print("DEVICE:", "GPU" if torch.cuda.is_available() else "CPU")
print("DATASET EXISTS:", dataset_path.exists())
print("CHARACTERS:", len(loaded_text))
print("STATUS:", "Stage 1 complete")

PROJECT: TinyGPT-English
PYTORCH: 2.11.0+cu130
DEVICE: GPU
DATASET EXISTS: True
CHARACTERS: 457
STATUS: Stage 1 complete


My own character level tokenizer

In [28]:


# Find every distinct character in the dataset
chars = sorted(set(text))

# create a mapping from character to integer ID
stoi = {ch: i for i, ch in enumerate(chars)}

# create the reverse mapping from integer ID to character
itos = {i: ch for i, ch in enumerate(chars)}

vocab_size = len(chars)


# Encoder and Decoder

def encode(input_text):
  unknown_chars = sorted(set(input_text) - set(stoi))

  if unknown_chars:
    raise ValueError(f"Unknown characters found: {unknown_chars}")
  return [stoi[ch] for ch in input_text]



def decode(token_ids):
  try:
    return "".join(itos[idx] for idx in token_ids)
  except KeyError as exc:
    raise ValueError(f"Invalid token ID: {exc.args[0]}") from exc

# convert the corpus into PyTorch tensor

import torch

encoded_text = encode(text)

data = torch.tensor(encoded_text,
dtype=torch.long)



# save the tokenizer configuration

import json

tokenizer_path = Path("/content/tinygpt_data/vocab.json")

tokenizer_config = {
    "type": "character",
    "vocab": chars,
    "vocab_size": vocab_size
}

tokenizer_path.write_text(
    json.dumps(tokenizer_config, ensure_ascii=False, indent=2), encoding="utf-8"
)





341

In [30]:
assert len(chars) > 0
assert len(stoi) == vocab_size
assert len(itos) == vocab_size
assert len(data) == len(text)
assert decode(data.tolist()) == text
assert tokenizer_path.exists()

print("=" * 35)
print("TinyGPT-English: stage 2")
print("=" * 35)
print("Vocabulary size:", vocab_size)
print("Encoded tokens:", len(data))
print("Tokenizer tests: PASSED")
print("Vocabulary file: SAVED")
print("STATUS: stage 2 completed")

TinyGPT-English: stage 2
Vocabulary size: 31
Encoded tokens: 457
Tokenizer tests: PASSED
Vocabulary file: SAVED
STATUS: stage 2 completed


Automated test for tokenizer

In [20]:
# Test 1: Basic encoding and decoding
sample = "hello world"
assert decode(encode(sample)) == sample
print("PASS: Text round-trip")

# Test 2: Entire dataset round-trip
all_ids = encode(text)
assert decode(all_ids) == text
print("PASS: Entire dataset round-trip")

# Test 3: Every character has a unique ID
assert len(stoi) == len(set(stoi.values()))
print("PASS: Unique token IDs")

# Test 4: Every ID maps back to its original character
for ch, idx in stoi.items():
  assert itos[idx] == ch
print("PASS: Reverse vocabulary mapping")

# Test 5: Empty text
assert encode("") == []
assert decode([]) == ""
print("PASS: Empty text")

# Test 6: Unknown characters should produce a clear Error
try:
  encode("&")
except ValueError:
  print("PASS: Unknown-character handling")
else:
  raise AssertionError("Unknown character was not rejected")

print("\nAll tokenizer tests passes")

PASS: Text round-trip
PASS: Entire dataset round-trip
PASS: Unique token IDs
PASS: Reverse vocabulary mapping
PASS: Empty text
PASS: Unknown-character handling

All tokenizer tests passes


Original dataset characters: 457
Encoded token count: 457
Tensor shape: torch.Size([457])
Tensor dtype: torch.int64

First 30 token IDS:
[3, 23, 25, 15, 12, 15, 9, 15, 7, 17, 0, 15, 19, 25, 11, 17, 17, 15, 13, 11, 19, 9, 11, 0, 15, 24, 0, 7, 0, 12]
